# 03 - Kinematic feature clustering

**Purpose**: understand how kinematic features relate to each other.
Cluster them by pairwise correlation, then view the same clusters
projected into PCA space.

**Input**: raw ``data.AKDdf`` restricted to contacted reaches, using
every numeric non-metadata column (including unit duplicates) so the
dendrogram reveals which measurements are functionally redundant.

Intuition target: confirm that ``prefer_calibrated_units`` is dropping
the right duplicates (same measurement in different units should
cluster together), and see which non-duplicate features carry
overlapping information.


In [ ]:
# parameters
N_CLUSTERS = 8        # Tune to where the dendrogram visually splits
FIGSIZE_DENDRO = (16, 6)
FIGSIZE_2D = (12, 10)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from endpoint_ck_analysis.config import EXAMPLE_OUTPUT_DIR, METADATA_COLS
from endpoint_ck_analysis.data_loader import load_all
from endpoint_ck_analysis.helpers.plotting import stamp_version

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
data = load_all()


## 1. Build the feature matrix

Contacted reaches only. Every numeric non-metadata column (including
unit duplicates) is retained so the dendrogram can reveal redundancy.


In [ ]:
contacted = data.AKDdf[data.AKDdf['contact_group'] == 'contacted']
kine_cols = [c for c in contacted.select_dtypes(include='number').columns if c not in METADATA_COLS]
feature_matrix = contacted[kine_cols].fillna(0)  # rows = reaches, columns = features

# Convert correlation to distance (highly correlated -> short distance).
# |r| treats negative correlations as similarity (same info, flipped sign).
corr = feature_matrix.corr()
dist = 1 - corr.abs()

link = linkage(pdist(dist), method='ward')  # Ward minimizes within-cluster variance

cluster_ids = fcluster(link, t=N_CLUSTERS, criterion='maxclust')
feature_clusters = pd.Series(cluster_ids, index=corr.columns, name='cluster')


## 2. Dendrogram


In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE_DENDRO)
dendrogram(link, labels=corr.columns.tolist(), leaf_rotation=90, ax=ax)
ax.set_ylabel('Clustering distance')
ax.set_title(f'Kinematic feature dendrogram (contacted reaches; cut at {N_CLUSTERS} clusters)')
plt.tight_layout()
stamp_version(fig, label='03 dendrogram')
plt.savefig(EXAMPLE_OUTPUT_DIR / '03_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()


## 3. PCA on the same feature matrix

Fit a feature-space PCA (features as the items, reaches as the
observations) so we can plot features in 2D/3D.


In [ ]:
X_scaled = StandardScaler().fit_transform(feature_matrix)
pca_feat = PCA(n_components=3)
pca_feat.fit(X_scaled)

loadings_xyz = pd.DataFrame(
    pca_feat.components_.T,
    index=feature_matrix.columns,
    columns=['PC1', 'PC2', 'PC3'],
)
loadings_xyz['cluster'] = feature_clusters


## 4. 2D view with one label per cluster

Labels land on the feature closest to each cluster's 2D centroid to
keep the plot legible.


In [ ]:
representatives = []
for cid, group in loadings_xyz.groupby('cluster'):
    centroid = group[['PC1', 'PC2']].mean()
    dists = ((group[['PC1', 'PC2']] - centroid) ** 2).sum(axis=1)
    representatives.append(dists.idxmin())

fig, ax = plt.subplots(figsize=FIGSIZE_2D)
scatter = ax.scatter(
    loadings_xyz['PC1'], loadings_xyz['PC2'],
    c=loadings_xyz['cluster'], cmap='tab10', alpha=0.8, s=60,
)
for feature in representatives:
    row = loadings_xyz.loc[feature]
    ax.annotate(feature, (row['PC1'], row['PC2']), fontsize=9, fontweight='bold')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel(f"PC1 ({pca_feat.explained_variance_ratio_[0]:.1%} variance)")
ax.set_ylabel(f"PC2 ({pca_feat.explained_variance_ratio_[1]:.1%} variance)")
ax.set_title(f'Kinematic features in PC1-PC2 space ({N_CLUSTERS} clusters)')
plt.colorbar(scatter, ax=ax, label='Cluster ID', ticks=range(1, N_CLUSTERS + 1))
plt.tight_layout()
stamp_version(fig, label='03 2D PCA')
plt.savefig(EXAMPLE_OUTPUT_DIR / '03_feature_clusters_2d.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nCluster membership ({N_CLUSTERS} clusters):')
for cid, group in loadings_xyz.groupby('cluster'):
    print(f'  Cluster {cid} ({len(group)} features): {sorted(group.index.tolist())}')


## 5. 3D view (plotly -- interactive)

Hover a dot to see its feature name; drag to rotate.
Not saved as a PNG because the interactivity is the point.


In [ ]:
fig3d = px.scatter_3d(
    loadings_xyz.reset_index().rename(columns={'index': 'feature'}),
    x='PC1', y='PC2', z='PC3',
    hover_name='feature',
    color=loadings_xyz['cluster'].astype(str).values,
    color_discrete_sequence=px.colors.qualitative.T10,
    title='Kinematic features in PC1-PC2-PC3 space (colored by cluster)',
    labels={
        'PC1': f"PC1 ({pca_feat.explained_variance_ratio_[0]:.1%})",
        'PC2': f"PC2 ({pca_feat.explained_variance_ratio_[1]:.1%})",
        'PC3': f"PC3 ({pca_feat.explained_variance_ratio_[2]:.1%})",
        'color': 'Cluster',
    },
)
fig3d.update_traces(marker=dict(size=5, opacity=0.85))
fig3d.show()


<!-- INTERPRETATION_BLOCK -->
## How to read this notebook's output

<details>
<summary>What the feature dendrogram + 2D/3D scatter tell you (click to expand)</summary>

**Dendrogram** (height = 1 - |correlation|): how kinematic features
relate to each other.

- Pairs of features at distance ~0 = essentially the same measurement
  (e.g., `max_extent_mm` and `max_extent_pixels` should sit at distance ~0
  -- this is the `prefer_calibrated_units` deduplication validating
  itself).
- Tight cluster of 3-4 features = those features carry overlapping
  information; you don't lose much by dropping all but one.
- Features sitting alone at the right edge = they carry independent
  information not captured by anything else in the dataset.
- Big chunks of the dendrogram with similar height = the kinematic
  feature space has clear functional groupings (e.g., velocity-related
  features clustering together separate from path-shape features).

**2D PC scatter colored by cluster + labeled centroids**: feature
relationships projected into PC space. Tight clusters in 2D = features
that can be summarized by a single new axis. Spread clusters = the
clustering is forcing structure that PCA does not see.

**3D plotly view**: same plot interactive; hover any dot to see the full
feature name. Use this when 2D labels collide.

The dendrogram is the most informative panel; the 2D/3D PC views are
sanity checks on the cluster structure.

</details>
